# 🗄️ GarudaChip Knowledge-Store Explorer

Browse and manage what's inside the RLM's long-term memory — **Postgres (pgvector)**
+ **MinIO object storage** — without touching the app. From here you can:

- **see** every row in Postgres and every blob in MinIO,
- **filter / select** by `kind` or `design`, read a row's full text, preview a blob,
- **semantic search** (the same recall the agent uses),
- **add** data manually (a note, a file, a whole folder),
- **remove** data (one item, or all of a kind/design),
- **re-export the committed seed** so your edits ship with the repo.

> Needs the stack up: `docker compose up -d`. Everything here goes through
> `MemoryStore` in `src/garuda_chip/memory_store.py`, so the notebook and the app
> see the exact same data.


## 0 · Connect

In [1]:
import os, sys
from pathlib import Path
# don't reach out to HuggingFace for model-card checks (recall uses the cached model)
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "garuda_chip" / "memory_store.py").exists()), Path.cwd())
sys.path.insert(0, str(REPO / "src" / "garuda_chip"))

from memory_store import MemoryStore, SEED_DIR
mem = MemoryStore()                     # bypasses auto-seed; just connects
print("Postgres :", "ready" if mem._db_ready else "OFFLINE", "—", mem.db_url.split("@")[-1])
print("MinIO    :", "ready" if mem._s3_ready else "OFFLINE", "—",
      mem.s3_endpoint, "bucket:", mem.s3_bucket)
print("stats    :", mem.stats())
assert mem.enabled, "Store offline — run `docker compose up -d` in the repo root first."

Postgres : ready — localhost:5433/garudachip
MinIO    : ready — http://localhost:9100 bucket: garudachip
stats    : {'total': 98, 'by_kind': {'design': 2, 'fix': 1, 'sim': 3, 'code': 92}}


## 1 · See what's in **Postgres**

Each row = catalog columns + a `vector(384)` embedding. The table below is metadata
only (no embedding / full text). Change `KIND` / `DESIGN` to filter, or set them to
`None` for everything.


In [2]:
import pandas as pd
KIND   = None      # e.g. "code", "fix", "design", "reference", "sim", "gds", "image"
DESIGN = None      # e.g. "generated_cnn"

rows = mem.list_items(kind=KIND, design=DESIGN, limit=500)
df = pd.DataFrame(rows)
print(f"{len(df)} row(s)")
df[["id", "kind", "design", "title", "has_vector", "object_key"]] if not df.empty else df

98 row(s)


,id,kind,design,title,has_vector,object_key
0,k_0aa98defc1863ff1,code,pipelined,cadd_pipelined.v,True,garuda/pipelined/cadd_pipelined.v
1,k_0578cc1d955a9666,code,pipelined,crot_gate_pipelined.v,True,garuda/pipelined/crot_gate_pipelined.v
2,k_d7d0e1d9edd8dee9,code,pipelined,sine_approx_pipelined.v,True,garuda/pipelined/sine_approx_pipelined.v
3,k_ce06dd5589667dd6,code,pipelined,qft3_top_pipelined.v,True,garuda/pipelined/qft3_top_pipelined.v
4,k_417e0a4f2327dd47,code,pipelined,ccmult_pipelined.v,True,garuda/pipelined/ccmult_pipelined.v
...,...,...,...,...,...,...
93,k_dff7efcebb6ac0b9,code,generated_fft,SdfUnit2.v,True,garuda/generated_fft/SdfUnit2.v
94,k_41ed8a1895397fb7,code,generated_pwm_generator,pwm_generator.v,True,garuda/generated_pwm_generator/pwm_generator.v
95,k_1293d8703cf05f34,design,pwm,pwm generator,True,NaN
96,k_2561a1f6ddaca5a0,design,relu,int8 relu lut,True,NaN


**Counts by kind / by design** — the shape of the store at a glance:

In [3]:
s = mem.stats()
print("total rows:", s.get("total"))
print("by kind   :", s.get("by_kind"))
import pandas as pd
all_rows = mem.list_items(limit=10000)
if all_rows:
    display(pd.DataFrame(all_rows)["design"].value_counts().head(20).rename("rows"))

total rows: 98
by kind   : {'design': 2, 'fix': 1, 'sim': 3, 'code': 92}


design
src                                              11
pipelined                                        10
tb                                               10
generated_qft_quantum_fourier_transform           9
generated_fft                                     8
pipeline_reduced_2                                7
final                                             7
pipeline_reduced_1                                6
generated_risc_v                                  6
generated_fir_filter                              4
generated_risc_8bit                               4
generated_sha256                                  4
generated_cnn                                     3
generated_iir_filter                              2
generated_pwm_generator                           2
generated_cordic                                  1
generated_8_bit_transformer_q_k_v_accelerator     1
pwm                                               1
relu                                              1
gemm 

**Read one full row** — paste an `id` from the table above:

In [4]:
ITEM_ID = rows[0]["id"] if rows else None      # <-- replace with any id

item = mem.get(ITEM_ID) if ITEM_ID else None
if item:
    for k in ("id", "kind", "design", "source", "title", "object_key", "tags", "created_at"):
        print(f"{k:11}: {item.get(k)}")
    print("\n--- text ---\n" + (item.get("text") or "")[:3000])
else:
    print("no such id")

id         : k_0aa98defc1863ff1
kind       : code
design     : pipelined
source     : corpus:qft/pipelined/cadd_pipelined.v
title      : cadd_pipelined.v
object_key : garuda/pipelined/cadd_pipelined.v
tags       : 
created_at : 2026-06-11 03:21:07.950839+00:00

--- text ---
`include "fixed_point_params.vh"

module cadd_pipelined(
    input                         clk,    // Clock
    input                         rst_n,  // Asynchronous reset, active-low
    input  signed [`TOTAL_WIDTH-1:0] ar, ai,   // Input A
    input  signed [`TOTAL_WIDTH-1:0] br, bi,   // Input B
    output signed [`ADD_WIDTH-1:0]   pr, pi    // Pipelined Output P
);

    reg signed [`ADD_WIDTH-1:0] pr_reg, pi_reg;

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            pr_reg <= 0;
            pi_reg <= 0;
        end else begin
            pr_reg <= ar + br;
            pi_reg <= ai + bi;
        end
    end

    assign pr = pr_reg;
    assign pi = pi_reg;

endmodule


## 2 · Semantic search (the agent's recall)

Embeds your query and orders by cosine distance in pgvector — exactly what the agent's
`recall_knowledge` / the retrieve step do. Optionally filter by `kind`.


In [5]:
QUERY = "riscv verilog code"
KIND_FILTER = None        # e.g. "fix" to only recall lessons, "code" for modules

for r in mem.recall(QUERY, kind=KIND_FILTER, k=8):
    print(f"  {r['score']:+.3f}  [{r['kind']}] {r['title']}  (design={r['design']}, key={r['object_key'] or '-'})")

/home/irman/GarudaChip/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12645.65it/s]


  +0.635  [code] risc_8bit_processor.v  (design=generated_risc_8bit, key=garuda/generated_risc_8bit/risc_8bit_processor.v)
  +0.600  [code] riscv_core.v  (design=generated_risc_v, key=garuda/generated_risc_v/riscv_core.v)
  +0.486  [code] SdfUnit.v  (design=generated_fft, key=garuda/generated_fft/SdfUnit.v)
  +0.485  [code] risc_8bit_processor_tb.v  (design=generated_risc_8bit, key=garuda/generated_risc_8bit/risc_8bit_processor_tb.v)
  +0.482  [code] reg_file.v  (design=generated_risc_v, key=garuda/generated_risc_v/reg_file.v)
  +0.471  [code] risc_8bit_defines.vh  (design=generated_risc_8bit, key=garuda/generated_risc_8bit/risc_8bit_defines.vh)
  +0.447  [code] cadd_pipelined.v  (design=pipelined, key=garuda/pipelined/cadd_pipelined.v)
  +0.444  [code] SdfUnit2.v  (design=generated_fft, key=garuda/generated_fft/SdfUnit2.v)


## 3 · See what's in **MinIO** (object storage)

The raw blobs behind the rows, keyed `garuda/<design>/<file>`. Filter with `PREFIX`.


In [6]:
import pandas as pd
PREFIX = ""        # e.g. "garuda/generated_cnn/"
objs = mem.list_objects(prefix=PREFIX)
print(f"{len(objs)} object(s), {sum(o['size'] for o in objs)/1024:.0f} KiB total")
pd.DataFrame(objs).sort_values("key") if objs else "（empty)"

94 object(s), 346 KiB total


,key,size
0,garuda/final/crot_pi_2_gate_pipelined.v,1798
1,garuda/final/crot_pi_4_gate_pipelined.v,2334
2,garuda/final/fixed_point_params.vh,559
3,garuda/final/h_gate_simplified.v,3579
4,garuda/final/qft3_top_pipelined.v,11615
...,...,...
89,garuda/tb/h_gate_tb.v,3978
90,garuda/tb/qft3_top_tb.v,3133
91,garuda/tb/sine_approx_tb.v,872
92,garuda/tb/swap_gate_tb.v,2138


**Preview / download one blob** — paste an object key from above:

In [ ]:
OBJECT_KEY = objs[0]["key"] if objs else None     # <-- replace with any key
SAVE_TO    = ""        # set a local path to also save it, e.g. "/tmp/out.v"

blob = mem.get_object(OBJECT_KEY) if OBJECT_KEY else None
if blob is None:
    print("no such object")
else:
    print(f"{OBJECT_KEY} — {len(blob)} bytes")
    if SAVE_TO:
        Path(SAVE_TO).write_bytes(blob); print("saved ->", SAVE_TO)
    ext = OBJECT_KEY.rsplit(".", 1)[-1].lower()
    if ext in ("png", "jpg", "jpeg", "webp", "gif"):
        from IPython.display import Image, display; display(Image(data=blob))
    else:
        print("\n--- content (first 2500 chars) ---\n" + blob.decode("utf-8", "replace")[:2500])

## 4 · **Add** data manually

Three ways, all go to Postgres (+ a vector) and — when a file is involved — MinIO.


**(a) a text note / lesson** (no file):

In [ ]:
rid = mem.remember(
    kind="fix",                                   # fix | note | design | reference | …
    text="When yosys errors 'unexpected [', the RTL used arrayed module ports; "
         "flatten them to packed vectors or split into separate ports.",
    design="general",
    source="manual",
    title="yosys: flatten arrayed ports",
    tags="synthesis",
)
print("added id:", rid)

**(b) a single file** (auto-detects kind; uploads the blob to MinIO):

In [ ]:
FILE = REPO / "examples" / "verilog_designs" / "generated_pwm_generator" / "pwm_generator.v"
rid = mem.ingest_file(FILE, design="generated_pwm_generator", source="manual")
print("ingested:", rid, "->", mem.get(rid)["object_key"] if rid else None)

**(c) a whole folder** — a finished design dir, or a reference corpus:

In [ ]:
# n = mem.ingest_run(REPO / "output" / "some_design", query="what it is")   # a design dir
# n = mem.ingest_corpus(REPO / "examples" / "verilog_designs")              # a corpus
# print("added", n, "items");
print("uncomment one of the lines above to bulk-add")

## 5 · **Remove** data manually

`delete(id)` removes one item (row **and** its MinIO blob). `delete_where(...)` bulk-deletes
by kind/design — it refuses to run with no filter, so you can't wipe everything by accident.


In [ ]:
# --- one item by id ---
DELETE_ID = ""                       # paste an id, then run
if DELETE_ID:
    print("deleted" if mem.delete(DELETE_ID) else "not found", DELETE_ID)

# --- bulk by kind and/or design (at least one required) ---
# n = mem.delete_where(design="generated_pwm_generator")
# n = mem.delete_where(kind="reference")
# print("deleted", n, "rows")
print("stats now:", mem.stats())

## 6 · Persist your edits into the **committed seed**

The seed (`data/knowledge_seed/`) is what auto-loads on a fresh clone. After adding /
removing items, re-export so your curated store ships with the repo (then commit it).


In [ ]:
n = mem.export_seed()
print(f"exported {n} rows + blobs to {SEED_DIR}")
print("commit it:  git add data/knowledge_seed && git commit -m 'update knowledge seed'")